@poleside 2025/11/4
read data from WW

In [2]:
import socket
import pandas as pd
import geopandas as gpd
import numpy as np
import os

### merge WW and RGI

In [2]:
# check rgi
import geopandas as gpd
rgi = r'C:\ML4GM\data\RGI7\RGI2000-v7.0-G-13_central_asia.shp'
gdf = gpd.read_file(rgi)
print(gdf.head())
print(len(gdf))
# print(gdf.columns)
# save as csv
output_csv_path = r'C:\ML4GM\data\RGI7\rgi_13.csv'
#    - index=False 表示在CSV中不保存 DataFrame 的索引（行号）
gdf.drop(columns='geometry').to_csv(output_csv_path, index=False)

                    rgi_id o1region o2region        glims_id  anlys_id  \
0  RGI2000-v7.0-G-13-00001       13    13-01  G067426E38743N    804440   
1  RGI2000-v7.0-G-13-00002       13    13-01  G067480E38714N    804446   
2  RGI2000-v7.0-G-13-00003       13    13-01  G067485E38713N    804448   
3  RGI2000-v7.0-G-13-00004       13    13-01  G067489E38714N    804451   
4  RGI2000-v7.0-G-13-00005       13    13-01  G067492E38714N    804453   

   subm_id             src_date     cenlon     cenlat  utm_zone  ...  \
0      752  2002-07-10T00:00:00  67.425881  38.743313        42  ...   
1      752  2002-07-10T00:00:00  67.479616  38.714583        42  ...   
2      752  2002-07-10T00:00:00  67.484971  38.713429        42  ...   
3      752  2002-07-10T00:00:00  67.489409  38.714494        42  ...   
4      752  2002-07-10T00:00:00  67.491937  38.713707        42  ...   

      zmin_m     zmax_m     zmed_m    zmean_m  slope_deg  aspect_deg  \
0  3693.8557  3783.9656  3727.2417  3728.6082  34.

In [3]:
# check rgi6
rgi = r'C:\ML4GM\data\RGI6\15_rgi60_SouthAsiaEast.shp'
gdf = gpd.read_file(rgi)
print(gdf.head())
print(len(gdf))
# print(gdf.columns)
# save as csv
output_csv_path = r'C:\ML4GM\data\RGI6\rgi_15.csv'
#    - index=False 表示在CSV中不保存 DataFrame 的索引（行号）
gdf.drop(columns='geometry').to_csv(output_csv_path, index=False)

            RGIId         GLIMSId   BgnDate   EndDate      CenLon     CenLat  \
0  RGI60-15.00001  G102044E29941N  19990920  -9999999  102.044042  29.941000   
1  RGI60-15.00002  G102042E29987N  19990920  -9999999  102.042346  29.987019   
2  RGI60-15.00003  G102041E29997N  19990920  -9999999  102.041130  29.997311   
3  RGI60-15.00004  G102050E29962N  19990920  -9999999  102.050283  29.962297   
4  RGI60-15.00005  G102044E30025N  19990920  -9999999  102.043728  30.025101   

  O1Region O2Region   Area  Zmin  ...  Aspect  Lmax  Status  Connect  Form  \
0       15        3  0.438  4996  ...     251   850       0        0     0   
1       15        3  0.644  4947  ...     244  1021       0        0     0   
2       15        3  0.225  5019  ...     274   812       0        0     0   
3       15        3  0.985  4622  ...      52  2318       0        0     0   
4       15        3  0.465  4733  ...      20   913       0        0     0   

   TermType  Surging  Linkages  Name  \
0         

In [ ]:
# ============================================================
# 步骤1：合并 RGI6 属性 与 WW 质量变化数据
# 筛选冰川：RGI6 ∩ WW（左连接，以 WW 为主）
# ============================================================
import pandas as pd

merged_data = []
print("开始处理冰川数据...")

for i in range(13, 16):
    print(f"\n--- 正在处理区域 {i} ---")
    try:
        df_rgi = pd.read_csv(f'C:/ML4GM/data/RGI6/rgi_{i}.csv')
        df_ww  = pd.read_csv(f'C:/ML4GM/data/glacierMass/dh_{i}_rgi60_pergla_rates.csv')

        n_rgi = df_rgi['RGIId'].nunique()
        n_ww  = df_ww['rgiid'].nunique()

        df_rgi['join_key'] = df_rgi['RGIId'].str[-5:]
        df_ww['join_key']  = df_ww['rgiid'].str[-5:]

        merged_df = pd.merge(df_ww, df_rgi, on='join_key', how='left')

        n_matched   = merged_df['RGIId'].notna().sum() // (len(merged_df) // n_ww or 1)
        n_unmatched = merged_df['RGIId'].isna().any()
        print(f"  RGI 冰川数: {n_rgi}  |  WW 冰川数: {n_ww}")
        if merged_df['RGIId'].isna().any():
            miss_ids = merged_df.loc[merged_df['RGIId'].isna(), 'rgiid'].unique()
            print(f"  ⚠ 有 {len(miss_ids)} 个 WW 冰川未匹配到 RGI 属性（前5个: {miss_ids[:5].tolist()}）")
        else:
            print(f"  ✓ 全部 WW 冰川均匹配到 RGI 属性")

        merged_data.append(merged_df)

    except FileNotFoundError as e:
        print(f"错误: 找不到文件 {e.filename}。请检查路径。")
    except Exception as e:
        print(f"处理区域 {i} 时发生错误: {e}")

if merged_data:
    ww_rgi = pd.concat(merged_data, ignore_index=True)
    print(f"\n--- 所有区域合并完成 ---")
    print(f"总行数: {len(ww_rgi)}  |  唯一冰川数: {ww_rgi['rgiid'].nunique()}")
    ww_rgi.to_csv('C:/ML4GM/data/ww_rgi.csv', index=False)
    print(f"已保存: C:/ML4GM/data/ww_rgi.csv")
else:
    print("没有处理任何数据，请检查文件路径")
    ww_rgi = pd.DataFrame()

In [ ]:
# [探索性代码，不参与主流程，已跳过]
# lat_min, lat_max, lon_min, lon_max = 27, 32, 92, 99
# slc_ww_rgi = ww_rgi[ww_rgi['CenLat'].between(lat_min, lat_max)
#                     & ww_rgi['CenLon'].between(lon_min, lon_max)]
# print(f"selected data count: {len(slc_ww_rgi)}")
# slc_ww_rgi.to_csv('C:/ML4GM/data/slc_ww_rgi.csv', index=False)

In [ ]:
# [探索性代码，不参与主流程，已跳过]
# print(slc_ww_rgi.head())

In [11]:
# count glaciers by rgi
total = 0
for i in range(13, 16):
    df_rgi = pd.read_csv(f'C:/ML4GM/data/RGI6/rgi_{i}.csv')

    lat_min = 27
    lat_max = 46
    lon_min = 67
    lon_max = 104
    slc_df_rgi = df_rgi[df_rgi['CenLat'].between(lat_min, lat_max)  
                        & df_rgi['CenLon'].between(lon_min, lon_max)]
    count = len(slc_df_rgi)
    print(f"rgi{i}: {count}")
    total = total + count
print(f"total glaciers: {total}")


rgi13: 54429
rgi14: 27988
rgi15: 13119
total glaciers: 95536


### process ERA5

In [1]:
# 必须先 import dask（且要在 xarray 之前），xarray 的 chunks= 才能用 dask
import dask
import dask.array
from dask.diagnostics import ProgressBar
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.transform import from_origin
from scipy.interpolate import RegularGridInterpolator
from netCDF4 import Dataset
from tqdm import tqdm  # 进度条显示
import os

In [ ]:
# ============================================================
# [已禁用] 双线性插值步骤
# 现在直接使用 ALL_14_ERA.nc（0.1° 原始分辨率），跳过插值。
# 如需恢复，取消下方注释并提供对应输入文件路径。
# ============================================================

# from scipy.interpolate import RegularGridInterpolator
# input_nc = r"C:\ML4GM\data\ERA5\ALL_t2mTp_ERA.nc"
# interpolated_nc = input_nc.replace('.nc', '_interpolated.nc')
# bilinear_interpolation_era5(input_nc, interpolated_nc)

In [13]:
import geopandas as gpd
import os

regions_to_count = [13, 14, 15]
base_shp_dir = r"C:\ML4GM\proc_data\03_RGI_A2"

region_counts = {}
total_count = 0

for region in regions_to_count:
    shp_path = os.path.join(base_shp_dir, f"{region}_A2.shp")
    if os.path.exists(shp_path):
        gdf = gpd.read_file(shp_path)
        num_elements = len(gdf)
        region_counts[region] = num_elements
        total_count += num_elements
        print(f"区域 {region} 的要素数量: {num_elements}")
    else:
        region_counts[region] = 0
        print(f"区域 {region} 的shp文件未找到 ({shp_path})")

print(f"总要素数量: {total_count}")

区域 13 的要素数量: 4197
区域 14 的要素数量: 2469
区域 15 的要素数量: 1435
总要素数量: 8101


In [15]:
# ============================================================
# 【A】计算月度气温递减率网格（逐年逐月，不取气候平均）
# 输入：PressureLevels_300_1000.nc（t, z 共 20 压力层）
# 输出：lapse_rate_grid.nc  dims=(valid_time, latitude, longitude)
#      valid_time 为每个年月（如 2000-01, 2000-02 ... 2019-12）
#      单位：K/m（典型值 −0.005 ~ −0.008）
# ============================================================
import xarray as xr
import numpy as np
from scipy import stats
import pandas as pd
import os

pl_path = r'C:\ML4GM\data\ERA5\PressureLevels_300_1000.nc'
out_nc  = r'C:\ML4GM\proc_data\01_glc_era\lapse_rate_grid.nc'
os.makedirs(os.path.dirname(out_nc), exist_ok=True)

G = 9.80665  # m/s²

ds_pl  = xr.open_dataset(pl_path)
T      = ds_pl['t']        # (valid_time, pressure_level, latitude, longitude)
Z      = ds_pl['z']        # geopotential m²/s²
alt    = Z / G             # 几何高度 m

T_np   = T.values          # (n_times, n_lev, nlat, nlon)
alt_np = alt.values        # (n_times, n_lev, nlat, nlon)
times  = T['valid_time'].values
n_times, n_lev, n_lat, n_lon = T_np.shape

print(f"时间步数: {n_times}  ({pd.Timestamp(times[0]).strftime('%Y-%m')} ~ {pd.Timestamp(times[-1]).strftime('%Y-%m')})")
print(f"网格: {n_lat}×{n_lon}，压力层: {n_lev}")

lr_arr = np.full((n_times, n_lat, n_lon), np.nan, dtype=np.float32)

for t in range(n_times):
    if t % 12 == 0:
        print(f"  处理 {pd.Timestamp(times[t]).strftime('%Y-%m')} ...")
    for i in range(n_lat):
        for j in range(n_lon):
            t_col   = T_np[t, :, i, j]
            alt_col = alt_np[t, :, i, j]
            if np.any(np.isnan(t_col)) or np.any(np.isnan(alt_col)):
                continue
            slope, *_ = stats.linregress(alt_col, t_col)
            lr_arr[t, i, j] = slope   # K/m

lats = T['latitude'].values
lons = T['longitude'].values
lr_da = xr.DataArray(
    lr_arr,
    dims=['valid_time', 'latitude', 'longitude'],
    coords={'valid_time': times, 'latitude': lats, 'longitude': lons},
    name='lapse_rate',
    attrs={'units': 'K/m', 'long_name': 'Monthly temperature lapse rate dT/dz'}
)
lr_da.to_netcdf(out_nc)
print(f"\n递减率网格已保存: {out_nc}  shape={lr_arr.shape}")
print(f"  值域: {np.nanmin(lr_arr):.5f} ~ {np.nanmax(lr_arr):.5f} K/m")

时间步数: 312  (2000-01 ~ 2025-12)
网格: 77×149，压力层: 20
  处理 2000-01 ...
  处理 2001-01 ...
  处理 2002-01 ...
  处理 2003-01 ...
  处理 2004-01 ...
  处理 2005-01 ...
  处理 2006-01 ...
  处理 2007-01 ...
  处理 2008-01 ...
  处理 2009-01 ...
  处理 2010-01 ...
  处理 2011-01 ...
  处理 2012-01 ...
  处理 2013-01 ...
  处理 2014-01 ...
  处理 2015-01 ...
  处理 2016-01 ...
  处理 2017-01 ...
  处理 2018-01 ...
  处理 2019-01 ...
  处理 2020-01 ...
  处理 2021-01 ...
  处理 2022-01 ...
  处理 2023-01 ...
  处理 2024-01 ...
  处理 2025-01 ...

递减率网格已保存: C:\ML4GM\proc_data\01_glc_era\lapse_rate_grid.nc  shape=(312, 77, 149)
  值域: -0.00811 ~ -0.00370 K/m


In [6]:
# ============================================================
# 【B】计算各冰川高程梯度 elev_grad = Zmed − ERA5格点海拔
# 输入：Geopotential.nc（ERA5-Land z 变量，时不变场）+ RGI shapefiles（Zmed）
# 输出：glacier_elev_grad.csv  列：RGIId, era5_altitude, elev_grad
#
# 方法（与论文一致）：
#   论文 Sec.2.3.1："We used the geopotential parameter from the ERA5-Land dataset
#   to determine the geometric altitude of each region."
#   ERA5-Land geopotential（z，m²/s²）为时不变地形场，除以 g 即得地形高度（m）。
# ============================================================
import xarray as xr
import geopandas as gpd
import pandas as pd
import numpy as np
import os

geo_path = r'C:\ML4GM\data\ERA5\Geopotential.nc'
base_shp = r'C:\ML4GM\proc_data\03_RGI_A2'
out_csv  = r'C:\ML4GM\proc_data\01_glc_era\glacier_elev_grad.csv'

G = 9.80665  # m/s²

# 加载 ERA5-Land 地形位势，去掉时间维，得到 (lat, lon) 静态场
ds_geo   = xr.open_dataset(geo_path)
z_static = ds_geo['z'].squeeze()          # (latitude, longitude)，m²/s²
alt_grid = z_static / G                   # 地形高度，m

records = []
for region in [13, 14, 15]:
    shp_path = os.path.join(base_shp, f'{region}_A2.shp')
    if not os.path.exists(shp_path):
        print(f'跳过区域 {region}: shapefile 不存在')
        continue
    gdf = gpd.read_file(shp_path)[['RGIId', 'Zmed', 'CenLon', 'CenLat']]
    print(f"区域 {region}: {len(gdf)} 个冰川...")

    for _, row in gdf.iterrows():
        era5_alt = float(alt_grid.sel(
            latitude=row['CenLat'],
            longitude=row['CenLon'],
            method='nearest'
        ).values)
        records.append({
            'RGIId':         row['RGIId'],
            'era5_altitude': round(era5_alt, 1),
            'elev_grad':     round(row['Zmed'] - era5_alt, 1)
        })

elev_df = pd.DataFrame(records)
elev_df.to_csv(out_csv, index=False)
print(f"\n高程梯度已保存: {out_csv}  ({len(elev_df)} 冰川)")
print(elev_df[['era5_altitude', 'elev_grad']].describe())

区域 13: 4197 个冰川...
区域 14: 2469 个冰川...
区域 15: 1435 个冰川...

高程梯度已保存: C:\ML4GM\proc_data\01_glc_era\glacier_elev_grad.csv  (8101 冰川)
       era5_altitude    elev_grad
count    8101.000000  8101.000000
mean     4770.666165   405.781928
std       652.167868   315.793541
min      2756.900000 -4207.900000
25%      4326.800000   217.400000
50%      4838.300000   375.600000
75%      5278.800000   568.400000
max      6321.800000  2687.600000


In [17]:
# ============================================================
# 【C】从递减率网格提取各冰川逐年逐月递减率（最近邻采样）
# 输出：lapse_rate_13/14/15.csv  列：RGIId, Year, Month, lapse_rate
# ============================================================
import xarray as xr
import geopandas as gpd
import pandas as pd
import os

lr_nc    = r'C:\ML4GM\proc_data\01_glc_era\lapse_rate_grid.nc'
base_shp = r'C:\ML4GM\proc_data\03_RGI_A2'
out_dir  = r'C:\ML4GM\proc_data\01_glc_era'

lr_da = xr.open_dataset(lr_nc)['lapse_rate']  # (valid_time, lat, lon)
times = pd.DatetimeIndex(lr_da['valid_time'].values)

for region in [13, 14, 15]:
    shp_path = os.path.join(base_shp, f'{region}_A2.shp')
    if not os.path.exists(shp_path):
        continue
    gdf = gpd.read_file(shp_path)[['RGIId', 'CenLon', 'CenLat']]
    records = []
    for _, row in gdf.iterrows():
        lr_ts = lr_da.sel(
            latitude=row['CenLat'],
            longitude=row['CenLon'],
            method='nearest'
        ).values   # shape (n_times,)
        for idx, t in enumerate(times):
            records.append({
                'RGIId':      row['RGIId'],
                'Year':       t.year,
                'Month':      t.month,
                'lapse_rate': float(lr_ts[idx])
            })
    lr_df = pd.DataFrame(records)
    lr_df.to_csv(os.path.join(out_dir, f'lapse_rate_{region}.csv'), index=False)
    print(f"区域 {region}: {len(lr_df)} 行  lapse_rate 均值={lr_df['lapse_rate'].mean():.5f} K/m")
    print(f"  年份范围: {lr_df['Year'].min()}–{lr_df['Year'].max()}")

区域 13: 1309464 行  lapse_rate 均值=-0.00643 K/m
  年份范围: 2000–2025
区域 14: 770328 行  lapse_rate 均值=-0.00623 K/m
  年份范围: 2000–2025
区域 15: 447720 行  lapse_rate 均值=-0.00586 K/m
  年份范围: 2000–2025


In [1]:
# ============================================================
# 步骤3：ERA5 地表变量提取（最近邻采样）
# 方法：按冰川质心坐标 sel(method='nearest')，与论文一致
# 输出：glcera_{13,14,15}.csv  列：RGIId, time, t2m, rsn, ...
# ============================================================
import xarray as xr
import geopandas as gpd
import pandas as pd
import os

nc_path      = r"C:\ML4GM\data\ERA5\ALL_14_ERA.nc"
variables    = ['t2m', 'rsn', 'sde', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'sshf', 'ssrd', 'strd', 'sp', 'tp']
base_shp_dir = r"C:\ML4GM\proc_data\03_RGI_A2"
base_out_dir = r"C:\ML4GM\proc_data\01_glc_era"
os.makedirs(base_out_dir, exist_ok=True)

print("正在加载 ERA5 数据集...")
ds = xr.open_dataset(nc_path, mask_and_scale=True)[variables].astype("float32")

for region in [13, 14, 15]:
    shp_path = os.path.join(base_shp_dir, f"{region}_A2.shp")
    out_csv  = os.path.join(base_out_dir,  f"glcera_{region}.csv")
    if not os.path.exists(shp_path):
        print(f"跳过区域 {region}：未找到 {shp_path}")
        continue

    gdf = gpd.read_file(shp_path)[['RGIId', 'CenLon', 'CenLat']]
    print(f"\n--- 区域 {region}：{len(gdf)} 个冰川 ---")

    rows = []
    for idx, row in gdf.iterrows():
        if (idx + 1) % 500 == 0 or idx == 0:
            print(f"  进度: {idx + 1}/{len(gdf)}")
        ts = ds.sel(latitude=row['CenLat'], longitude=row['CenLon'], method='nearest')
        tmp = ts.to_dataframe().reset_index()
        time_col = [c for c in tmp.columns if 'time' in c.lower()][0]
        tmp = tmp.rename(columns={time_col: 'time'})
        tmp = tmp[['time'] + [v for v in variables if v in tmp.columns]]
        tmp['RGIId'] = row['RGIId']
        rows.append(tmp)

    df_out = pd.concat(rows, ignore_index=True)
    df_out = df_out[['RGIId', 'time'] + [v for v in variables if v in df_out.columns]]
    df_out.to_csv(out_csv, index=False)
    print(f"  已保存: {out_csv}  (冰川数: {df_out['RGIId'].nunique()}, 行数: {len(df_out)})")

正在加载 ERA5 数据集...

--- 区域 13：4197 个冰川 ---
  进度: 1/4197
  进度: 500/4197
  进度: 1000/4197
  进度: 1500/4197
  进度: 2000/4197
  进度: 2500/4197
  进度: 3000/4197
  进度: 3500/4197
  进度: 4000/4197
  已保存: C:\ML4GM\proc_data\01_glc_era\glcera_13.csv  (冰川数: 4197, 行数: 1309464)

--- 区域 14：2469 个冰川 ---
  进度: 1/2469
  进度: 500/2469
  进度: 1000/2469
  进度: 1500/2469
  进度: 2000/2469
  已保存: C:\ML4GM\proc_data\01_glc_era\glcera_14.csv  (冰川数: 2469, 行数: 770328)

--- 区域 15：1435 个冰川 ---
  进度: 1/1435
  进度: 500/1435
  进度: 1000/1435
  已保存: C:\ML4GM\proc_data\01_glc_era\glcera_15.csv  (冰川数: 1435, 行数: 447720)


In [2]:
# 转换数据格式（长表 → 宽表，支持全部 14 个 ERA5 变量）
import pandas as pd
import os

ERA5_VARS = ['t2m', 'rsn', 'sde', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'sshf', 'ssrd', 'strd', 'sp', 'tp']

input_dir  = r"C:\ML4GM\proc_data\01_glc_era"
output_dir = r"C:\ML4GM\proc_data\01_glc_era"
os.makedirs(output_dir, exist_ok=True)
regions = [13, 14, 15]

for region in regions:
    input_path = os.path.join(input_dir, f"glcera_{region}.csv")
    if not os.path.exists(input_path):
        print(f"跳过区域 {region}：未找到 {input_path}")
        continue
    print(f"转换区域 {region}...")
    df = pd.read_csv(input_path)

    df['time']  = pd.to_datetime(df['time'])
    df['Year']  = df['time'].dt.year
    df['Month'] = df['time'].dt.month

    # 对每个存在的变量做 pivot，然后拼接
    pivots = []
    for var in ERA5_VARS:
        if var not in df.columns:
            print(f"  警告：变量 {var} 不在数据中，跳过")
            continue
        piv = df.pivot(index=['RGIId', 'Year'], columns='Month', values=var)
        piv.columns = [f"{m}_{var}" for m in piv.columns]
        pivots.append(piv)

    df_transformed = pd.concat(pivots, axis=1).reset_index()

    # --- 追加气温递减率宽表（12列，逐年逐月）---
    lr_long_path = os.path.join(input_dir, f'lapse_rate_{region}.csv')
    if os.path.exists(lr_long_path):
        lr_long = pd.read_csv(lr_long_path)
        # lr_long 列：RGIId, Year, Month, lapse_rate → pivot 按 (RGIId, Year) 为索引
        piv_lr = lr_long.pivot(index=['RGIId', 'Year'], columns='Month', values='lapse_rate')
        piv_lr.columns = [f'{m}_lr' for m in piv_lr.columns]
        df_transformed = df_transformed.merge(piv_lr.reset_index(), on=['RGIId', 'Year'], how='left')
        n_null = df_transformed[[f'{m}_lr' for m in range(1, 13)]].isna().any(axis=1).sum()
        if n_null > 0:
            print(f"  ⚠ {n_null} 行缺少 lapse_rate（年份不在压力层数据范围内）")
        else:
            print(f"  已合并 lapse_rate 宽表（12列，逐年变化）")

    output_path = os.path.join(output_dir, f"glcera_{region}_transformed.csv")
    df_transformed.to_csv(output_path, index=False)
    print(f"  已保存: {output_path}  ({len(df_transformed)} 行, {len(df_transformed.columns)} 列)")
    print(df_transformed.head())

转换区域 13...
  已合并 lapse_rate 宽表（12列，逐年变化）
  已保存: C:\ML4GM\proc_data\01_glc_era\glcera_13_transformed.csv  (109122 行, 182 列)
            RGIId  Year      1_t2m      2_t2m      3_t2m      4_t2m  \
0  RGI60-13.00062  2000  244.42763  242.51756  248.22067  253.89601   
1  RGI60-13.00062  2001  242.16945  244.89401  248.92113  256.78546   
2  RGI60-13.00062  2002  241.60558  244.96110  249.03098  255.34792   
3  RGI60-13.00062  2003  244.36570  245.56325  247.81680  254.76566   
4  RGI60-13.00062  2004  243.82907  245.05595  251.04625  255.73933   

       5_t2m      6_t2m      7_t2m      8_t2m  ...      3_lr      4_lr  \
0  265.47192  269.99414  274.19897  270.71118  ... -0.006770 -0.006761   
1  264.52338  269.02734  274.02320  272.78370  ... -0.006853 -0.006738   
2  259.44430  267.28687  272.00366  273.24683  ... -0.006676 -0.006696   
3  255.54485  265.50390  271.84302  272.81616  ... -0.006593 -0.006621   
4  258.57635  265.94165  271.26172  271.67334  ... -0.006689 -0.006584   

     

In [20]:
# ============================================================
# 步骤2：筛选年度数据 + Area ≥ 2 km²
# 去除原因：非年度时段（如2年/5年均值）、过小冰川
# ============================================================
import pandas as pd

def process_glacier_annual_data(input_path, output_path):
    print(f"正在读取文件: {input_path} ...")
    df = pd.read_csv(input_path, low_memory=False)
    n_raw = len(df)

    def extract_year_if_annual(period_str):
        try:
            start_date, end_date = str(period_str).split('_')
            s_year, e_year = int(start_date[:4]), int(end_date[:4])
            if start_date.endswith('-01-01') and end_date.endswith('-01-01') and (e_year == s_year + 1):
                return s_year
            return None
        except:
            return None

    df['year'] = df['period'].apply(extract_year_if_annual)
    df_annual = df[df['year'].notnull()].copy()
    n_after_annual = len(df_annual)
    n_drop_period = n_raw - n_after_annual
    print(f"  原始行数: {n_raw}")
    print(f"  去除非年度时段（2年/5年均值等）: -{n_drop_period} 行 → {n_after_annual} 行")

    if 'Area' in df_annual.columns:
        n_before_area = len(df_annual)
        df_annual = df_annual[df_annual['Area'] >= 2].copy()
        n_drop_area = n_before_area - len(df_annual)
        n_glc_before = df[df['year'].notnull()]['rgiid'].nunique()
        n_glc_after  = df_annual['rgiid'].nunique()
        print(f"  去除 Area < 2 km² 的冰川: -{n_drop_area} 行 ({n_glc_before - n_glc_after} 个冰川) → {len(df_annual)} 行")
    else:
        print("  警告：未找到 'Area' 列，跳过面积筛选。")

    df_annual['year'] = df_annual['year'].astype(int)
    cols = list(df_annual.columns)
    if 'period' in cols:
        period_idx = cols.index('period')
        cols.pop(cols.index('year'))
        cols.insert(period_idx, 'year')
        df_annual = df_annual[cols].drop(columns=['period'])

    df_annual.to_csv(output_path, index=False)
    print(f"\n  最终: {len(df_annual)} 行，{df_annual['rgiid'].nunique()} 个冰川")
    print(f"  已保存: {output_path}")

input_file  = r"C:\ML4GM\data\ww_rgi.csv"
output_file = r"C:\ML4GM\proc_data\02_merge\ww_rgi_formed.csv"
process_glacier_annual_data(input_file, output_file)

正在读取文件: C:\ML4GM\data\ww_rgi.csv ...
  原始行数: 4012512
  去除非年度时段（2年/5年均值等）: -2101792 行 → 1910720 行
  去除 Area < 2 km² 的冰川: -1748700 行 (87435 个冰川) → 162020 行

  最终: 162020 行，8101 个冰川
  已保存: C:\ML4GM\proc_data\02_merge\ww_rgi_formed.csv


In [3]:
# ============================================================
# 步骤4：合并三区 ERA5 宽表，筛选 2000–2019 年
# ============================================================
import pandas as pd
import os

def merge_and_filter_glcera(input_paths, output_path):
    existing = [p for p in input_paths if os.path.exists(p)]
    missing  = [p for p in input_paths if not os.path.exists(p)]
    if missing:
        print(f"  ⚠ 以下文件不存在，跳过: {missing}")
    if not existing:
        raise FileNotFoundError(f"未找到任何输入文件")

    dfs = []
    for p in existing:
        df = pd.read_csv(p)
        print(f"  读取 {os.path.basename(p)}: {df['RGIId'].nunique()} 个冰川，{len(df)} 行")
        dfs.append(df)

    df_merged = pd.concat(dfs, ignore_index=True)
    n_before  = df_merged['RGIId'].nunique()

    df_filtered = df_merged[(df_merged['Year'] >= 2000) & (df_merged['Year'] <= 2019)].copy()
    n_after = df_filtered['RGIId'].nunique()
    if n_before != n_after:
        print(f"  ⚠ 年份筛选后丢失冰川: {n_before} → {n_after}（-{n_before - n_after}）")
    else:
        print(f"  ✓ 年份筛选（2000–2019）未丢失任何冰川，共 {n_after} 个")

    df_filtered = df_filtered.sort_values(by=['RGIId', 'Year'])
    df_filtered.to_csv(output_path, index=False)
    print(f"  已保存: {output_path}  ({len(df_filtered)} 行)")

base_dir = r"C:\ML4GM\proc_data\01_glc_era"
input_paths = [
    os.path.join(base_dir, "glcera_13_transformed.csv"),
    os.path.join(base_dir, "glcera_14_transformed.csv"),
    os.path.join(base_dir, "glcera_15_transformed.csv"),
]
output_file = r"C:\ML4GM\proc_data\02_merge\glcera_yeared_merged.csv"
os.makedirs(os.path.dirname(output_file), exist_ok=True)
merge_and_filter_glcera(input_paths, output_file)

  读取 glcera_13_transformed.csv: 4197 个冰川，109122 行
  读取 glcera_14_transformed.csv: 2469 个冰川，64194 行
  读取 glcera_15_transformed.csv: 1435 个冰川，37310 行
  ✓ 年份筛选（2000–2019）未丢失任何冰川，共 8101 个
  已保存: C:\ML4GM\proc_data\02_merge\glcera_yeared_merged.csv  (162020 行)


In [4]:
# ============================================================
# 步骤5：合并 WW 年度数据 与 ERA5 气候宽表（内连接）
# 丢失原因：ERA5 提取时部分冰川无有效格点覆盖
# ============================================================
import pandas as pd
import os

def merge_glacier_and_climate_data(ww_path, glc_path, output_path):
    print("正在加载数据...")
    df_ww  = pd.read_csv(ww_path)
    df_glc = pd.read_csv(glc_path)

    if 'Year' in df_glc.columns:
        df_glc = df_glc.rename(columns={'Year': 'year'})

    df_ww['year']  = df_ww['year'].astype(int)
    df_glc['year'] = df_glc['year'].astype(int)

    n_ww_glc  = df_ww['RGIId'].nunique()
    n_era_glc = df_glc['RGIId'].nunique()

    # 找出在 WW 中有但 ERA5 中没有的冰川
    ww_ids  = set(df_ww['RGIId'].unique())
    era_ids = set(df_glc['RGIId'].unique())
    missing_in_era = ww_ids - era_ids
    if missing_in_era:
        print(f"  ⚠ ERA5 缺少以下 {len(missing_in_era)} 个冰川（WW 有但 ERA5 无）:")
        miss_df = df_ww[df_ww['RGIId'].isin(missing_in_era)][['RGIId', 'CenLon', 'CenLat']].drop_duplicates()
        print(miss_df.to_string(index=False))
    else:
        print(f"  ✓ WW 与 ERA5 冰川完全覆盖，无缺失")

    print(f"\n  WW 冰川数: {n_ww_glc}  |  ERA5 冰川数: {n_era_glc}")
    print("正在进行数据合并（内连接）...")
    df_merged = pd.merge(df_ww, df_glc, on=['RGIId', 'year'], how='inner')
    n_merged_glc = df_merged['RGIId'].nunique()
    print(f"  合并后保留冰川: {n_merged_glc}  (-{n_ww_glc - n_merged_glc} 个冰川因 ERA5 缺失被丢弃)")

    # 追加高程梯度（静态列）
    elev_path = r'C:\ML4GM\proc_data\01_glc_era\glacier_elev_grad.csv'
    if os.path.exists(elev_path):
        elev_df_tmp = pd.read_csv(elev_path)[['RGIId', 'elev_grad']]
        df_merged   = pd.merge(df_merged, elev_df_tmp, on='RGIId', how='left')
        n_null_elev = df_merged['elev_grad'].isna().sum()
        if n_null_elev > 0:
            print(f"  ⚠ {n_null_elev} 行缺少 elev_grad")
        else:
            print(f"  ✓ 高程梯度合并完成（非空 {df_merged['elev_grad'].notna().sum()} 行）")

    df_merged.to_csv(output_path, index=False)
    print(f"\n  最终行数: {len(df_merged)}  |  冰川数: {df_merged['RGIId'].nunique()}")
    print(f"  已保存: {output_path}")

ww_file     = r"C:\ML4GM\proc_data\02_merge\ww_rgi_formed.csv"
glc_file    = r"C:\ML4GM\proc_data\02_merge\glcera_yeared_merged.csv"
output_file = r"C:\ML4GM\proc_data\02_merge\merged_data.csv"
merge_glacier_and_climate_data(ww_file, glc_file, output_file)

正在加载数据...
  ✓ WW 与 ERA5 冰川完全覆盖，无缺失

  WW 冰川数: 8101  |  ERA5 冰川数: 8101
正在进行数据合并（内连接）...
  合并后保留冰川: 8101  (-0 个冰川因 ERA5 缺失被丢弃)
  ✓ 高程梯度合并完成（非空 162020 行）

  最终行数: 162020  |  冰川数: 8101
  已保存: C:\ML4GM\proc_data\02_merge\merged_data.csv


In [5]:
# 删除不需要的列
df = pd.read_csv(output_file)
columns_to_drop = ['area', 'err_dhdt', 'dvoldt', 'err_dvoldt', 'dmdt', 'err_dmdt', 'dmdtda', 
                   'err_dmdtda', 'perc_area_meas', 'perc_area_res', 'valid_obs', 'valid_obs_py', 
                   'reg', 'join_key', 'RGIId', 'BgnDate', 'EndDate', 'O1Region', 'O2Region', 
                   'Status', 'Connect', 'Form', 'TermType', 'Surging', 'Linkages', 'Name']

df_cleaned = df.drop(columns=columns_to_drop)
# 根据output_file路径，改个名字保存
cleaned_output_file = output_file.replace('.csv', '_cleaned.csv')
df_cleaned.to_csv(cleaned_output_file, index=False)
df_cleaned.head()

,rgiid,year,dhdt,GLIMSId,CenLon,CenLat,Area,Zmin,Zmax,Zmed,...,4_lr,5_lr,6_lr,7_lr,8_lr,9_lr,10_lr,11_lr,12_lr,elev_grad
0,RGI60-13.00062,2000,0.4247,G078112E35641N,78.1118,35.6407,2.222,5417,6011,5839,...,-0.006761,-0.006329,-0.005641,-0.005117,-0.005463,-0.006048,-0.006668,-0.006656,-0.006366,227.4
1,RGI60-13.00062,2001,0.4973,G078112E35641N,78.1118,35.6407,2.222,5417,6011,5839,...,-0.006738,-0.006471,-0.005699,-0.005118,-0.005366,-0.006206,-0.006697,-0.006542,-0.006457,227.4
2,RGI60-13.00062,2002,0.3004,G078112E35641N,78.1118,35.6407,2.222,5417,6011,5839,...,-0.006696,-0.006475,-0.005937,-0.005319,-0.005180,-0.006222,-0.006801,-0.006748,-0.006450,227.4
3,RGI60-13.00062,2003,0.1291,G078112E35641N,78.1118,35.6407,2.222,5417,6011,5839,...,-0.006621,-0.006654,-0.005919,-0.005355,-0.005297,-0.005860,-0.006823,-0.006653,-0.006484,227.4
4,RGI60-13.00062,2004,0.4655,G078112E35641N,78.1118,35.6407,2.222,5417,6011,5839,...,-0.006584,-0.006437,-0.005949,-0.005439,-0.005473,-0.005857,-0.006630,-0.006675,-0.006395,227.4
